In [1]:
#imports 
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from dtreeviz import model
from sklearn.tree import plot_tree
from sklearn.model_selection import GridSearchCV

In [2]:
df=pd.read_csv("MBA.csv")
print(df.index)
print("All columns are")
ai=0
for i in df:
    print(ai,i)
    ai+=1

RangeIndex(start=0, stop=6194, step=1)
All columns are
0 application_id
1 gender
2 international
3 gpa
4 major
5 race
6 gmat
7 work_exp
8 work_industry
9 admission


In [3]:
df.shape

(6194, 10)

In [4]:
df.head()

,application_id,gender,international,gpa,major,race,gmat,work_exp,work_industry,admission
0,1,Female,False,3.30,Business,Asian,620.0,3.0,Financial Services,Admit
1,2,Male,False,3.28,Humanities,Black,680.0,5.0,Investment Management,NaN
2,3,Female,True,3.30,Business,NaN,710.0,5.0,Technology,Admit
3,4,Male,False,3.47,STEM,Black,690.0,6.0,Technology,NaN
4,5,Male,False,3.35,STEM,Hispanic,590.0,5.0,Consulting,NaN


In [5]:
# Filling missing values

# Filling missing race with unknown
df["race"] = df["race"].fillna("Unknown")



# Filling missing admission value with Rejected
df["admission"] = df["admission"].fillna("Rejected")

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6194 entries, 0 to 6193
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   application_id  6194 non-null   int64  
 1   gender          6194 non-null   object 
 2   international   6194 non-null   bool   
 3   gpa             6194 non-null   float64
 4   major           6194 non-null   object 
 5   race            6194 non-null   object 
 6   gmat            6194 non-null   float64
 7   work_exp        6194 non-null   float64
 8   work_industry   6194 non-null   object 
 9   admission       6194 non-null   object 
dtypes: bool(1), float64(3), int64(1), object(5)
memory usage: 441.7+ KB


In [7]:
# Dropping application_id
df = df.drop(['application_id'],axis=1)

In [8]:
# Label Encoding
label_encoders = {}

for col in df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

In [9]:
df.sample(n=10)

,gender,international,gpa,major,race,gmat,work_exp,work_industry,admission
1804,1,False,2.99,1,5,570.0,5.0,3,1
3762,1,False,3.46,1,5,740.0,4.0,6,1
3161,1,False,3.33,1,5,710.0,6.0,11,0
936,1,True,3.24,0,4,570.0,4.0,1,1
729,1,False,2.89,1,0,580.0,4.0,13,1
5749,1,False,3.46,2,0,670.0,5.0,1,1
4986,1,False,3.55,2,5,720.0,5.0,10,1
6050,1,False,3.42,1,5,670.0,5.0,5,0
6180,1,False,3.30,0,0,600.0,5.0,10,1
6151,1,False,3.16,0,0,610.0,4.0,10,1


In [10]:
# Decision Tree
dt = DecisionTreeClassifier(random_state=42)

In [11]:
X = df.drop(columns=["admission"])
y = df["admission"]

In [12]:
# Train Test Split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

In [13]:
# parameter grid

paramgrid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 7, 10,],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": [None, "sqrt", "log2"],
    "class_weight": [None, "balanced"]
}

In [14]:
# Grid Search with 5-fold cross-validation

gridsearch = GridSearchCV(
    estimator=dt,
    param_grid=paramgrid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)


In [15]:
# Fit on training data

gridsearch.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'class_weight': [None, 'balanced'],
                         'criterion': ['gini', 'entropy'],
                         'max_depth': [3, 5, 7, 10],
                         'max_features': [None, 'sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 4, 8],
                         'min_samples_split': [2, 5, 10, 20]},
             scoring='accuracy')

In [16]:
results = pd.DataFrame(gridsearch.cv_results_)


paramsdf = results["params"].apply(pd.Series)


finalresults = pd.concat(
    [paramsdf,
     results[["mean_test_score", "std_test_score", "rank_test_score"]]],
    axis=1
)


finalresults = finalresults.sort_values("rank_test_score")

print(finalresults)

    class_weight criterion  max_depth max_features  min_samples_leaf  \
85          None      gini          5         log2                 2   
84          None      gini          5         log2                 2   
81          None      gini          5         log2                 1   
89          None      gini          5         log2                 4   
88          None      gini          5         log2                 4   
..           ...       ...        ...          ...               ...   
418     balanced      gini          3         log2                 1   
423     balanced      gini          3         log2                 2   
422     balanced      gini          3         log2                 2   
430     balanced      gini          3         log2                 8   
419     balanced      gini          3         log2                 1   

     min_samples_split  mean_test_score  std_test_score  rank_test_score  
85                   5         0.841776        0.001615     

In [17]:
finalresults.head(n=20)

,class_weight,criterion,max_depth,max_features,min_samples_leaf,min_samples_split,mean_test_score,std_test_score,rank_test_score
85,None,gini,5,log2,2,5,0.841776,0.001615,1
84,None,gini,5,log2,2,2,0.841776,0.001615,1
81,None,gini,5,log2,1,5,0.841776,0.001615,1
89,None,gini,5,log2,4,5,0.841776,0.002058,4
88,None,gini,5,log2,4,2,0.841776,0.002058,4
227,None,entropy,3,log2,1,20,0.841372,0.000404,6
201,None,entropy,3,None,4,5,0.841372,0.000404,6
202,None,entropy,3,None,4,10,0.841372,0.000404,6
203,None,entropy,3,None,4,20,0.841372,0.000404,6
204,None,entropy,3,None,8,2,0.841372,0.000404,6


In [18]:
# Training Best Model

bdt = DecisionTreeClassifier(
    criterion = "gini",
    max_depth = 5,
    min_samples_split = 2,
    min_samples_leaf = 2,
    max_features = "log2",
    class_weight = None
)

In [19]:
bdt.fit(X_train, y_train)

y_pred = bdt.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8256658595641646

Classification Report:
              precision    recall  f1-score   support

           0       0.25      0.01      0.01       196
           1       0.83      1.00      0.90      1025
           2       0.00      0.00      0.00        18

    accuracy                           0.83      1239
   macro avg       0.36      0.33      0.30      1239
weighted avg       0.72      0.83      0.75      1239


Confusion Matrix:
[[   1  195    0]
 [   3 1022    0]
 [   0   18    0]]


C:\Users\abhinavkuriya\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
C:\Users\abhinavkuriya\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
C:\Users\abhinavkuriya\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
